<img src="https://github.com/thesps/conifer/blob/master/conifer_v1.png?raw=true" width="250" alt="conifer" />

In this notebook we will take the first steps with training a BDT with `xgboost`, then translating it to HLS code for FPGA with `conifer`

Key concepts:
- dataset creation
- model training
- model evaluation
- `conifer` configuration and conversion
- model emulation / evaluation
- model synthesis

In [ ]:
from sklearn.datasets import make_moons
from sklearn.inspection import DecisionBoundaryDisplay
import xgboost as xgb
import matplotlib.pyplot as plt
import numpy as np
from scipy.special import expit
import conifer
import json
import os
import sys

# enable more output from conifer
import logging
logging.basicConfig(stream=sys.stdout, level=logging.WARNING)
logger = logging.getLogger('conifer')
logger.setLevel('DEBUG')

# create a random seed at we use to make the results repeatable
seed = int('fpga_tutorial'.encode('utf-8').hex(), 16) % 2**31

print(f'Using conifer version {conifer.__version__}')

# Create a dataset
For this first introduction we will create a synthetic random separable dataset with 2 variables from `scikit-learn` `make_moons`. You'll see where it gets the name when we plot it below. We make a crude train/test split, using some of the data for the model training and reserving the rest for testing.

In [ ]:
X, y = make_moons(n_samples=5000, noise=0.3, random_state=seed)
X_train, X_test = X[:2500], X[2500:]
y_train, y_test = y[:2500], y[2500:]

# save to files
os.makedirs('moons_dataset', exist_ok=True)
np.save('moons_dataset/X_train.npy', X_train)
np.save('moons_dataset/X_test.npy', X_test)
np.save('moons_dataset/y_train.npy', y_train)
np.save('moons_dataset/y_test.npy', y_test)

## Plot dataset
Visualise the dataset we just created. There are two variables that we plot on `(x,y)` and two classes that we show with colour.

In [ ]:
plt.scatter(X_train[:,0], X_train[:,1], c=y_train, cmap='PiYG', edgecolors='k')

# Train a BDT
We'll use `xgboost`'s `XGBClassifier` with:

| Parameter | Explanation |
| --- | --- |
| `n_estimators=30` | 30 trees |
| `max_depth=5` | maximum tree depth of 5 |
| `learning_rate=1.0` |  |

In [ ]:
clf = xgb.XGBClassifier(n_estimators=30, max_depth=5, learning_rate=1.0,
                        random_state=seed).fit(X_train, y_train)

# Validate performance
Now we check whether the trained model is any good. Firstly we'll visualise the decision boundary which shows how the strength of the prediction varies across the parameter space. Where there is more overlap between the classes, the prediction is less certain.

We plot a few of the test set examples on top.

In [ ]:
DecisionBoundaryDisplay.from_estimator(clf, X_test, cmap='PiYG')
plt.scatter(X_test[:,0][:200], X_test[:,1][:200], c=y_test[:200], cmap='PiYG', edgecolors='k')

<img src="https://github.com/thesps/conifer/blob/master/conifer_v1.png?raw=true" width="250" alt="conifer" />

Now we'll convert this model to FPGA firmware with `conifer`. We first need to create a configuration in the form of a dictionary. The quickest way to get started is to create a default configuration from the intended target backend (`xilinxhls` for us). Each backend may have different configuration options, so getting the configuration this way helps enumerate the possible options.

We will print the configuration, modify it, and print it again. The modifications are:
- set the `OutputDirectory` to something descriptive
- set the `XilinxPart` to the part number of the FPGA on the Alveo U50

In [ ]:
cfg = conifer.backends.xilinxhls.auto_config()

# print the config
print('Default Configuration\n' + '-' * 50)
print(json.dumps(cfg, indent=2))
print('-' * 50)

# modify the config
cfg['OutputDir'] = 'prj_conifer_part_1'
# target an fpga used in the CMS Level 1 Trigger Phase 2
cfg['XilinxPart'] = 'xcvu13p-flga2577-2-e'
# target a = 2.5 ns clock period (400 MHz clock frequency)
cfg['ClockPeriod'] = 2.5

# print the config again
print('Modified Configuration\n' + '-' * 50)
print(json.dumps(cfg, indent=2))
print('-' * 50)

## Convert and write
Convert the `xgboost` model to a `conifer` one, and print the `help` to see what methods it implements.
Then `write` the model, creating the specified output directory and writing all the HLS files to it. We also save the `xgboost` model itself.

#### Other converters:
`conifer` has converters for several popular BDT training libraries. Each one is used like: `conifer.converters.convert_from_<library>(model, config)`
The converters are:
- `sklearn`
- `xgboost`
- `ydf`
- `tmva`
- `onnx` (exposing `catboost` and `lightGBM`)

In [ ]:
# convert the model to the conifer representation
conifer_model = conifer.converters.convert_from_xgboost(clf, cfg)
# print the help to see the API on the conifer_model
help(conifer_model)
# write the project (writing HLS project to disk)
conifer_model.write() 
# save the conifer model - we can load this again later
clf.save_model('prj_conifer_part_1/xgboost_model.json')

## Explore
Browse the files in the newly created project directory to take a look at the HLS code.

The output of `!tree prj_conifer_part_1` is:

```
prj_conifer_part_1/
├── bridge.cpp
├── build_hls.tcl
├── firmware
│   ├── BDT.cpp
│   ├── BDT.h
│   ├── my_prj.cpp
│   ├── my_prj.h
│   └── parameters.h
├── hls_parameters.tcl
├── my_prj.json
├── my_prj_test.cpp
├── tb_data
└── vivado_synth.tcl

2 directories, 11 files
```

- files under `firmware` are the HLS implementation of the model
- `my_prj.json` is the saved converted `conifer` model that can be loaded again without the original `xgboost` model
- `tcl` scripts are used for synthesizing the project

## Emulate
Before starting the lengthy FPGA build process, we should validate that our conversion was successful and that the choice of precision was suitable. To do this we need to run the HLS C++ code on the CPU with some test data first. This is like the HLS C Simulation step, but rather than writing a C++ testbench and invoking `vitis_hls` to run `csim`, `conifer` implements Python bindings for the HLS.

We first need to compile (which uses the C++ compiler), then we can make predictions

In [ ]:
conifer_model.compile()

In [ ]:
y_hls = conifer_model.decision_function(X_test)

## Compare
Now we check whether the emulated predictions are good. To do this, first we'll make the decision boundary for both the `conifer` and `xgboost` models, and also show the difference.

In [ ]:
# make a 1000x1000 grid of points in the feature space
X_mesh = np.meshgrid(np.linspace(-3, 3, 1000), np.linspace(-3, 3, 1000))
# reshape them for inference
X_grid = np.vstack([X_mesh[0].ravel(), X_mesh[1].ravel()]).T

# run emulated inference, compute the class probability, reshape to 1000x1000 grid
y_hls = conifer_model.decision_function(X_grid)          # compute inference with conifer HLS emulation
y_hls_proba = expit(y_hls)                               # compute probabilities (external to conifer)
y_hls_mesh = np.reshape(y_hls_proba, X_mesh[0].shape)   # reshape

# run the xgboost prediction on the same grid
y_xgb_proba = clf.predict_proba(X_grid)[:,1]             # compute probabilities with xgboost
y_xgb_mesh = np.reshape(y_xgb_proba, X_mesh[0].shape)    # reshape

# compute the residual between conifer HLS and xgboost
y_residual = y_hls_proba - y_xgb_proba

In [ ]:
# display the boundaries, and the difference
f, axs = plt.subplots(1, 4, figsize=(20,5))

# plot HLS
display = DecisionBoundaryDisplay(xx0=X_mesh[0], xx1=X_mesh[1], response=y_hls_mesh)
display.plot(cmap='PiYG', ax=axs[0])
axs[0].scatter(X_test[:,0][:200], X_test[:,1][:200], c=y_test[:200], cmap='PiYG', edgecolors='k')
axs[0].set_title('HLS')

# plot the XGBoost
display = DecisionBoundaryDisplay(xx0=X_mesh[0], xx1=X_mesh[1], response=y_xgb_mesh)
display.plot(cmap='PiYG', ax=axs[1])
axs[1].scatter(X_test[:,0][:200], X_test[:,1][:200], c=y_test[:200], cmap='PiYG', edgecolors='k')
axs[1].set_title('XGBoost')

# plot the difference
pcm = axs[2].pcolormesh(X_mesh[0], X_mesh[1], y_xgb_mesh-y_hls_mesh)
axs[2].set_title('XGBoost - HLS')
f.colorbar(pcm)

x_lim = np.max(np.abs(y_residual))
h, b = np.histogram(y_residual, bins=np.linspace(-x_lim, +x_lim, 100))
b_width = b[1]-b[0]
axs[3].bar(b[:-1]+b_width/2, h, width=b_width)
axs[3].semilogy()
axs[3].set_title('XGBoost - HLS')

plt.tight_layout()

## Profile

The agreement between `xgboost` and `conifer` is good but it could be better. Now we'll `profile` the model to check if we can make a better choice than the default `ap_fixed<18,8>` precision. This plots the summary statistics of the distribution of thresholds for each variable and of leave scores. We need to choose a precision for the thresholds that covers most of the range of the respective plot, and a separate precision for the scores that covers most of the range of the leaf scores. Since the x-axis of the plot is displayed with base-2 logarithm, we can read off the bitwidth and integer bits for the `ap_fixed`

For the purposes of time we won't go through and change the configuration, but this profiling tool is the entry point if you need to choose the precision for a new model.

In [ ]:
conifer_model.profile()

## Build
Now we'll run the Vitis HLS and Vivado synthesis. HLS C Synthesis compiles our C++ to RTL, performing scheduling and resource mapping. Vivado synthesis synthesizes the RTL from the previous step into a netlist, and produces a more realistic resource estimation. The latency can't change during Vivado synthesis, it's fixed in the RTL description.

After the build completes we can also browse the new log files and reports that are generated.

In [ ]:
conifer_model.build(synth=True, vsynth=True)

## Report
If the synthesis completed successfuly, we can extract the key metrics from the reports and print them out.
The section `"vsynth"` contains the report from the Vivado RTL synthesis, which is usually lower, and more realistic than the HLS report.

In [ ]:
report = conifer_model.read_report()
print(json.dumps(report, indent=2))

## Accelerator

In this section we'll deploy the model that we already trained to a `pynq-z2` board.
We'll use the `AcceleratorConfig` part of the configuration that we previously left undefined.

In [ ]:
pynq_model_cfg = conifer.backends.xilinxhls.auto_config()
pynq_model_cfg['OutputDir'] = 'prj_conifer_part_1_pynq'          # choose a new project directory
pynq_model_cfg['ProjectName'] = 'conifer_xgboost_moons'
pynq_model_cfg['AcceleratorConfig'] = {'Board' : 'pynq-z2',      # choose a pynq-z2 board
                                       'InterfaceType' : 'float' # floating point for the data I/O (this is default)
                                      }

# print the config
print('Modified Configuration\n' + '-' * 50)
print(json.dumps(pynq_model_cfg, indent=2))
print('-' * 50)

## Supported boards

Here we print the list of supported boards, so you can see what else works "out of the box". It's relatively easy to add other Zynq SoC or Alveo boards, for example to add an Alveo U50 card targeting `xilinx_u50_gen3x16_xdma_5_202210_1` platform:

```
u50 = conifer.backends.boards.AlveoConfig.default_config()
u50['xilinx_part'] = 'xcu50-fsvh2104-2-e'
u50['platform'] = 'xilinx_u50_gen3x16_xdma_5_202210_1'
u50['name'] = 'xilinx_u50_gen3x16_xdma_5_202210_1'
u50 = conifer.backends.boards.AlveoConfig(u50)
conifer.backends.boards.register_board_config(u50.name, u50)
```

In [ ]:
# This is the full list of supported boards:
print(f'Supported boards: {conifer.backends.boards.get_available_boards()}')

### Load the model

We load the JSON for the conifer model we previously used, applying the new configuration just defined. We'll see that the FPGA part specified by the board overrides the `XilinxPart` specified in the default.

In [ ]:
pynq_model = conifer.model.load_model('prj_conifer_part_1/my_prj.json', new_config=pynq_model_cfg)
pynq_model.write()

## Build the model

Now we run `build` again, running HLS Synthesis, Logic Synthesis and Place & Route, finally producing a bitfile and an archive of files that we'll need to run inference on the pynq-z2 board. The floorplan of the bitfile should like something like this, where the individual tree modules are highlighted in different colours:

<img src="./images/pynq_z2_floorplan.png" width="300" />

In [ ]:
pynq_model.build(synth=True, bitfile=True, package=True)

# Inference on pynq-z2

Now we go to run inference on the pynq-z2 device, transferring:
- `prj_conifer_part_1_pynq/conifer_xgboost_moons.zip`
- `prj_conifer_part_1_pynq/conifer_xgboost_moons.json`

Then we return here with the inference results:
- `y_mesh_pynq_hls.npy`
- `y_mesh_pynq_fpu.npy`

...

In [ ]:
# load the files that we produced on the pynq

y_mesh_pynq_hls = np.load('y_mesh_pynq_hls.npy')
y_mesh_pynq_hls = np.reshape(expit(y_mesh_pynq_hls), X_mesh[0].shape)

y_mesh_pynq_fpu = np.load('y_mesh_pynq_fpu.npy')
y_mesh_pynq_fpu = np.reshape(expit(y_mesh_pynq_fpu), X_mesh[0].shape)

In [ ]:
# display the boundaries, and the difference
f, axs = plt.subplots(1, 3, figsize=(20,5))

# plot HLS (pynq inference)
display = DecisionBoundaryDisplay(xx0=X_mesh[0], xx1=X_mesh[1], response=y_mesh_pynq_hls)
display.plot(cmap='PiYG', ax=axs[0])
axs[0].scatter(X_test[:,0][:200], X_test[:,1][:200], c=y_test[:200], cmap='PiYG', edgecolors='k')
axs[0].set_title('HLS (pynq-z2 accelerator)')

# plot FPU (pynq inference)
display = DecisionBoundaryDisplay(xx0=X_mesh[0], xx1=X_mesh[1], response=y_mesh_pynq_fpu)
display.plot(cmap='PiYG', ax=axs[1])
axs[1].scatter(X_test[:,0][:200], X_test[:,1][:200], c=y_test[:200], cmap='PiYG', edgecolors='k')
axs[1].set_title('HLS (pynq-z2 accelerator)')

# plot the XGBoost
display = DecisionBoundaryDisplay(xx0=X_mesh[0], xx1=X_mesh[1], response=y_xgb_mesh)
display.plot(cmap='PiYG', ax=axs[2])
axs[2].scatter(X_test[:,0][:200], X_test[:,1][:200], c=y_test[:200], cmap='PiYG', edgecolors='k')
axs[2].set_title('XGBoost')